In [3]:
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2


URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)


In [4]:
def parse_gtfs_feed(response):
    """
    Parse a GTFS-RT response into a FeedMessage.

    Parameters
    ----------
    response : requests.Response
        HTTP response containing the serialized GTFS-RT feed.

    Returns
    -------
    FeedMessage
        Parsed GTFS-RT feed.
    """
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    print(f"Number of entities: {len(feed.entity)}")

    return feed

feed = parse_gtfs_feed(response)


Number of entities: 22534


In [5]:
for entity in feed.entity[:10]:
    print(entity)

id: "1173029tu"
trip_update {
  trip {
    trip_id: "1173029"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788558360
    }
    stop_id: "378754"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 1
    arrival {
      delay: 60
      time: 1788558480
    }
    departure {
      delay: 480
      time: 1788559260
    }
    stop_id: "38102"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 2
    arrival {
      delay: 540
      time: 1788559620
    }
    departure {
      delay: 660
      time: 1788559740
    }
    stop_id: "688994"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 3
    arrival {
      delay: 660
      time: 1788559860
    }
    stop_id: "457646"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 4
    arrival {
      delay: 720
      time: 178

In [6]:
stops_df = pd.read_csv("../data/mvv_stops.csv", delimiter=";")


In [7]:
def create_stop_name_mapping(stops_df):
    """
    Create a mapping from MVV stop IDs to stop names.

    Parameters
    ----------
    stops_df : pandas.DataFrame
        DataFrame containing the MVV stop data. It must contain
        the columns "HstNummer" and "Name ohne Ort".

    Returns
    -------
    dict
        Dictionary mapping stop IDs to stop names.
    """
    stops_df["HstNummer"] = stops_df["HstNummer"].astype(str)

    stop_names = (
        stops_df
        .set_index("HstNummer")["Name ohne Ort"]
        .to_dict()
    )

    return stop_names


In [8]:
def parse_trip_updates(feed, stop_names, trip_lines):
    """
    Parse GTFS-RT trip updates into a pandas DataFrame.

    Parameters
    ----------
    feed : FeedMessage
        Parsed GTFS-RT feed containing trip updates.
    stop_names : dict
        Mapping from stop IDs to stop names.
    trip_lines : dict
        Mapping from trip IDs to line names.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing trip, line, stop, arrival,
        departure, and delay information.
    """
    rows = []

    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue

        trip = entity.trip_update.trip

        # Get line for this trip
        line = trip_lines.get(str(trip.trip_id))

        # Ignore trips that are not part of the selected agencies
        if line is None:
            continue

        for stop in entity.trip_update.stop_time_update:

            row = {
                "trip_id": trip.trip_id,
                "start_date": trip.start_date,
                "line": line,
                "stop_id": str(stop.stop_id),
                "stop_name": stop_names.get(str(stop.stop_id)),
                "stop_sequence": stop.stop_sequence,
            }

            if stop.HasField("departure"):
                row["departure_time"] = datetime.fromtimestamp(
                    stop.departure.time
                )
                row["departure_delay"] = stop.departure.delay

            if stop.HasField("arrival"):
                row["arrival_time"] = datetime.fromtimestamp(
                    stop.arrival.time
                )
                row["arrival_delay"] = stop.arrival.delay

            rows.append(row)

    return pd.DataFrame(rows)

In [9]:
def preprocess_gtfs(data_dir, munich_agencies):
    """
    Preprocess GTFS static data for selected agencies.

    Returns
    -------
    trip_lines : dict
        Mapping from trip_id to line name.

    stop_names : dict
        Mapping from stop_id to stop name.
    """

    routes_df = pd.read_csv(f"{data_dir}/routes.txt")
    trips_df = pd.read_csv(f"{data_dir}/trips.txt")
    stops_df = pd.read_csv(f"{data_dir}/stops.txt")

    routes_df["route_id"] = routes_df["route_id"].astype(str)
    routes_df["agency_id"] = routes_df["agency_id"].astype(str)

    trips_df["trip_id"] = trips_df["trip_id"].astype(str)
    trips_df["route_id"] = trips_df["route_id"].astype(str)

    stops_df["stop_id"] = stops_df["stop_id"].astype(str)

    munich_routes = routes_df[
        routes_df["agency_id"].isin(munich_agencies)
    ]

    route_lines = (
        munich_routes
        .set_index("route_id")["route_short_name"]
        .to_dict()
    )

    munich_trips = trips_df[
        trips_df["route_id"].isin(route_lines)
    ]

    trip_lines = (
        munich_trips
        .set_index("trip_id")["route_id"]
        .map(route_lines)
        .to_dict()
    )

    stop_names = (
        stops_df
        .set_index("stop_id")["stop_name"]
        .to_dict()
    )

    return trip_lines, stop_names


In [10]:
munich_agencies = ["100", "191", "364"]

trip_lines, stop_names = preprocess_gtfs(
    "../data",
    munich_agencies
)

In [11]:
df = parse_trip_updates(
    feed,
    stop_names,
    trip_lines
)

df.head(100)

,trip_id,start_date,line,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,1622647,20260904,183,341638,Trabrennbahn,0,2026-09-05 00:01:27,-3.0,NaT,NaN
1,1622647,20260904,183,138548,Schichtlstraße,1,2026-09-05 00:02:46,16.0,2026-09-05 00:02:46,16.0
2,1622647,20260904,183,139540,Kunihohstraße,2,2026-09-05 00:03:31,31.0,2026-09-05 00:03:31,31.0
3,1622647,20260904,183,500127,Daglfing Bahnhof Ost,3,2026-09-05 00:05:03,33.0,2026-09-05 00:05:03,33.0
4,1622647,20260904,183,649905,Daglfing Bahnhof West,4,2026-09-05 00:05:33,-27.0,2026-09-05 00:05:33,3.0
...,...,...,...,...,...,...,...,...,...,...
95,1581156,20260904,55,453940,Pfanzeltplatz,10,2026-09-05 00:28:02,2.0,2026-09-05 00:27:09,-51.0
96,1581156,20260904,55,268392,Wilhelm-Hoegner-Straße,11,2026-09-05 00:28:56,-4.0,2026-09-05 00:28:56,-4.0
97,1581156,20260904,55,418397,Friedhof Perlach,12,2026-09-05 00:29:17,-13.0,2026-09-05 00:29:17,-13.0
98,1581156,20260904,55,41712,Thomas-Dehler-Straße,13,2026-09-05 00:30:13,-17.0,2026-09-05 00:30:13,-17.0


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5800 entries, 0 to 5799
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   trip_id          5800 non-null   object        
 1   start_date       5800 non-null   object        
 2   line             5800 non-null   object        
 3   stop_id          5800 non-null   object        
 4   stop_name        5800 non-null   object        
 5   stop_sequence    5800 non-null   int64         
 6   departure_time   5268 non-null   datetime64[ns]
 7   departure_delay  5268 non-null   float64       
 8   arrival_time     4970 non-null   datetime64[ns]
 9   arrival_delay    4970 non-null   float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(5)
memory usage: 453.2+ KB
